# 11 — LPPLS Robustness: filtracja, pokrycie i stabilność inicjalizacji

**Cel:** sprawdzić, jak zmiana założeń filtracji LPPLS wpływa na: liczbę zaakceptowanych okien, odsetek dni z sygnałem, zgodność z epizodami `UNION01–UNION14` i stabilność oszacowań. 

**Wejście wymagane:** `lppls_rolling_results.csv` ze wszystkimi oknami (bez wcześniejszej filtracji). Alternatywnie `lppls_rolling_all_fits_2015_2024.csv`. **Cena BTC jest wymagana wyłącznie w opcjonalnym eksperymencie powtórnych dopasowań.**

**Istotne ograniczenia:**
- Wyniki zgodności z `UNION` są *retrospektywną analizą wrażliwości*, nie testem prognozowania przyszłych spadków ani niezależnym OOS. Nie dobieraj progów na całym zbiorze, aby później oceniać je na tych samych datach.
- Jedno zapisane dopasowanie na okno **nie pozwala** ocenić zmienności względem inicjalizacji. Dlatego taki test ma własny, **opcjonalny** etap wielokrotnych dopasowań.
- `CI = liczba zaakceptowanych okien kończących się danego dnia / wszystkie okna kończące się tego dnia`; nie odrzucamy dni z zerem zaakceptowanych dopasowań.
- Warunki filtracji są wyliczane ponownie z surowych wartości parametrów, a nie z zapisanych flag `cond_*`.

Notatnik **nie** trenuje nowych modeli ML. Porównuje *sygnały LPPLS*, a nie przewagi XGBoost/LightGBM.

In [ ]:
import sys as _sys
from pathlib import Path as _ResearchPath
_sys.path.insert(0,str(_ResearchPath('.').resolve()/'src'))
from lppls_research import TemporalAvailabilityValidator
from pathlib import Path
from itertools import product
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import average_precision_score, roc_auc_score

INPUT_CANDIDATES = [
    Path('lppls_rolling_results.csv'),
    Path('lppls_rolling_results_z_etykietami_baniek.csv'),
]
OUT = Path('11_wyniki_robustness')
OUT.mkdir(parents=True, exist_ok=True)

# Wartości porównawcze — NIE są wybrane jako najlepsze na zbiorze testowym.
R2_LEVELS = [0.80, 0.85, 0.90, 0.95]
OMEGA_RANGES = [(6.0, 13.0), (5.0, 15.0)]
TC_MAX_LEVELS = [20, 40, 60]
M_RANGES = [(0.0, 1.0), (0.1, 0.9)]
BASELINE = dict(r2_min=0.90, w_lo=6.0, w_hi=13.0,
                tc_max=40, m_lo=0.0, m_hi=1.0)

# Kosztowny etap powtórnych dopasowań: domyślnie WYŁĄCZONY.
RUN_REFIT_STABILITY = False  # Set True after installing lppls; never claim unperformed stability tests.
PRICE_FILE = Path('btc_usd_2015_2024.csv')
REPEAT_SEEDS = [11, 22, 33, 44, 55]
REFIT_WINDOWS_PER_CLASS = 12   # 12 zaakceptowanych + 12 odrzuconych
REFIT_MAX_SEARCHES = 50

print('Wyniki zapisywane w:', OUT.resolve())

## 1. Wczytanie i kontrola kompletności dopasowań

In [ ]:
source = next((p for p in INPUT_CANDIDATES if p.exists()), None)
if source is None:
    raise FileNotFoundError('Nie znaleziono surowego CSV LPPLS. Sprawdzono: '
                            + ', '.join(map(str, INPUT_CANDIDATES)))

fits = pd.read_csv(source, parse_dates=['win_start', 'win_end'])
required = {'win_start', 'win_end', 'n', 'ok', 'tc', 'm', 'w', 'B', 'r2'}
missing = required.difference(fits.columns)
if missing:
    raise ValueError(f'CSV nie ma wymaganych kolumn: {sorted(missing)}')

fits['win_start'] = pd.to_datetime(fits.win_start).dt.normalize()
fits['win_end'] = pd.to_datetime(fits.win_end).dt.normalize()
if fits[['win_start','win_end']].isna().any().any():
    raise ValueError('Nieprawidłowe daty okien LPPLS')
if fits.duplicated(['win_start','win_end','n']).any():
    raise ValueError('Powielone identyfikatory okien: win_start, win_end, n')
if (fits.win_start > fits.win_end).any():
    raise ValueError('Data początku okna jest późniejsza niż koniec')

for col in ['tc','m','w','B','r2']:
    fits[col] = pd.to_numeric(fits[col], errors='coerce')

fits['t_gap'] = fits['tc'] - fits['win_end'].map(pd.Timestamp.toordinal).astype(float)
fit_ok = fits['ok'].astype(str).str.lower().eq('true').to_numpy()
valid_parameters = np.isfinite(fits[['tc','m','w','B','r2']].to_numpy()).all(axis=1)
valid = fit_ok & valid_parameters

# Wszystkie możliwe daty predykcji od pierwszego do ostatniego win_end.
dates = pd.date_range(fits.win_end.min(), fits.win_end.max(), freq='D', name='date')
date_codes = dates.get_indexer(fits.win_end)
assert (date_codes >= 0).all()
n_all = np.bincount(date_codes, minlength=len(dates)).astype(int)
assert n_all.sum() == len(fits)

# Odróżniamy dzień bez rozpatrywanych okien od dnia z 0 zaakceptowanych.
print(f'Plik: {source} | dopasowania: {len(fits):,} | dni: {len(dates):,}')
print('Dopasowania bez wyjątku i z kompletnymi parametrami:', int(valid.sum()))
print('Dni bez jakiegokolwiek okna:', int((n_all == 0).sum()))
print('Rok pierwszego okna:', fits.win_end.min().date(),
      '| ostatni:', fits.win_end.max().date())
display(fits.head(3))

## 2. Etykiety referencyjne UNION (wyłącznie do diagnostyki retrospektywnej)

In [ ]:
data_union = [
    ['UNION01','2016-06-08','2016-06-18'],
    ['UNION02','2016-12-30','2017-01-15'],
    ['UNION03','2017-02-22','2017-03-07'],
    ['UNION04','2017-04-27','2017-07-05'],
    ['UNION05','2017-07-15','2018-01-09'],
    ['UNION06','2019-06-17','2019-07-13'],
    ['UNION07','2019-08-09','2019-08-10'],
    ['UNION08','2019-12-06','2020-01-05'],
    ['UNION09','2020-02-07','2020-02-17'],
    ['UNION10','2020-03-20','2020-09-21'],
    ['UNION11','2020-11-06','2021-05-20'],
    ['UNION12','2021-08-08','2021-09-19'],
    ['UNION13','2021-10-05','2021-12-02'],
    ['UNION14','2024-03-04','2024-03-25'],
]
episodes = pd.DataFrame(data_union, columns=['bubble_id','start','end'])
episodes['start'] = pd.to_datetime(episodes['start'])
episodes['end'] = pd.to_datetime(episodes['end'])
reference = np.zeros(len(dates), dtype=np.int8)
for ep in episodes.itertuples(index=False):
    reference |= np.asarray((dates >= ep.start) & (dates <= ep.end), dtype=np.int8)
print('Referencyjne dni bańkowe:', int(reference.sum()),
      '| poza oznaczonymi epizodami:', int((reference == 0).sum()))

## 3. Filtr bazowy i kontrola zgodności ze starymi flagami

In [ ]:
def accept_mask(r2_min=0.90, w_lo=6, w_hi=13,
                tc_max=40, m_lo=0, m_hi=1):
    return (
        valid
        & (fits['m'].to_numpy() > m_lo)
        & (fits['m'].to_numpy() < m_hi)
        & (fits['w'].to_numpy() >= w_lo)
        & (fits['w'].to_numpy() <= w_hi)
        & (fits['B'].to_numpy() < 0)
        & (fits['t_gap'].to_numpy() > 0)
        & (fits['t_gap'].to_numpy() <= tc_max)
        & (fits['r2'].to_numpy() > r2_min)
    )

base_mask = accept_mask(**BASELINE)
base_counts = np.bincount(date_codes, weights=base_mask.astype(int),
                          minlength=len(dates)).astype(int)
base_days = base_counts > 0
print('BAZA:', int(base_mask.sum()), 'zaakceptowanych okien;',
      int(base_days.sum()), 'dni z sygnałem')
print('Akceptacja (%):', round(100*base_mask.mean(), 3))

# Flagi historyczne nie są używane do filtracji — służą tylko diagnostyce.
flag_mismatches = {}
for flag, actual in [
    ('cond_m', (fits.m > 0) & (fits.m < 1)),
    ('cond_w', fits.w.between(6,13)),
    ('cond_B', fits.B < 0),
    ('cond_tc', (fits.t_gap > 0) & (fits.t_gap <= 40))
]:
    if flag in fits.columns:
        saved = fits[flag].astype(str).str.lower().eq('true')
        flag_mismatches[flag] = int((saved.to_numpy() != actual.to_numpy()).sum())
print('Rozbieżności między zapisanymi i przeliczonymi flagami:', flag_mismatches)

## 4. Pełna siatka wrażliwości filtrów — liczba okien, CI i metryki dzienne

In [ ]:
def classification_metrics(y, alert, score):
    y = np.asarray(y, dtype=int)
    alert = np.asarray(alert, dtype=int)
    score = np.asarray(score, dtype=float)
    tp = int(((y == 1) & (alert == 1)).sum())
    fp = int(((y == 0) & (alert == 1)).sum())
    fn = int(((y == 1) & (alert == 0)).sum())
    tn = int(((y == 0) & (alert == 0)).sum())
    prec = tp/(tp+fp) if (tp+fp) else np.nan
    rec = tp/(tp+fn) if (tp+fn) else np.nan
    f1 = 2*tp/(2*tp+fp+fn) if (tp+fn) else np.nan
    spec = tn/(tn+fp) if (tn+fp) else np.nan
    ap = average_precision_score(y,score) if y.sum() else np.nan
    roc = roc_auc_score(y,score) if np.unique(y).size == 2 else np.nan
    return dict(TP=tp, FP=fp, FN=fn, TN=tn, precision=prec, recall=rec,
                f1=f1, specificity=spec, average_precision=ap, roc_auc=roc)

records = []
annual_records = []
# Nie porównujemy wariantów na różnych dniach — dla każdego cały wspólny okres.
for r2_min, (w_lo,w_hi), tc_max, (m_lo,m_hi) in product(
        R2_LEVELS, OMEGA_RANGES, TC_MAX_LEVELS, M_RANGES):
    cfg = dict(r2_min=r2_min, w_lo=w_lo, w_hi=w_hi,
               tc_max=tc_max, m_lo=m_lo, m_hi=m_hi)
    cid = f'r2_{r2_min:.2f}_w_{w_lo:g}_{w_hi:g}_tc_{tc_max}_m_{m_lo:g}_{m_hi:g}'
    accepted = accept_mask(**cfg)
    n_acc = np.bincount(date_codes, weights=accepted.astype(int),
                        minlength=len(dates)).astype(int)
    ci = np.divide(n_acc, n_all, out=np.zeros(len(dates), dtype=float),
                   where=n_all > 0)
    alert = (n_acc > 0).astype(int)
    cm = classification_metrics(reference, alert, ci)
    events_hit = int(sum(alert[(dates >= ep.start) & (dates <= ep.end)].any()
                         for ep in episodes.itertuples(index=False)))
    records.append(dict(config_id=cid, **cfg, n_all_windows=len(fits),
        accepted_windows=int(accepted.sum()), acceptance_pct=100*accepted.mean(),
        days_with_signal=int(alert.sum()), coverage_pct=100*alert.mean(),
        bubble_days=int(reference.sum()), bubble_episode_hits=events_hit,
        positive_rate=float(reference.mean()), **cm))
    for year in sorted(dates.year.unique()):
        take = (dates.year == year)
        mtr = classification_metrics(reference[take], alert[take], ci[take])
        annual_records.append(dict(config_id=cid, year=int(year),
                                   n_days=int(take.sum()),
                                   alarm_days=int(alert[take].sum()), **mtr))

sensitivity = pd.DataFrame(records)
annual = pd.DataFrame(annual_records)
sensitivity.to_csv(OUT/'11_01_sensitivity_grid.csv', index=False)
annual.to_csv(OUT/'11_02_sensitivity_by_year.csv', index=False)

base_id = 'r2_0.90_w_6_13_tc_40_m_0_1'
base_row = sensitivity.set_index('config_id').loc[base_id]
print('Porównano wariantów:',len(sensitivity))
print('\nBAZA:')
display(base_row.to_frame().T[[
    'accepted_windows','acceptance_pct','days_with_signal','coverage_pct',
    'precision','recall','f1','average_precision','bubble_episode_hits'
]])
print('\nNajmniejszy i największy zasięg sygnałów w siatce:')
display(sensitivity[['accepted_windows','days_with_signal','precision','recall']].describe())

## 5. Wykresy wrażliwości filtracji

In [ ]:
# Porównanie R² przy pozostałych parametrach identycznych jak w bazie.
r2_slice = sensitivity.loc[
    (sensitivity.w_lo == 6) & (sensitivity.w_hi == 13)
    & (sensitivity.tc_max == 40)
    & (sensitivity.m_lo == 0) & (sensitivity.m_hi == 1)
].sort_values('r2_min')

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(r2_slice.r2_min, r2_slice.accepted_windows, marker='o', label='Okna')
axes[0].set(xlabel='Minimalne R² (warunek >)', ylabel='Zaakceptowane okna',
            title='Wpływ R² na liczbę okien')
axes[1].plot(r2_slice.r2_min, r2_slice.coverage_pct, marker='o')
axes[1].set(xlabel='Minimalne R² (warunek >)', ylabel='Dni z sygnałem [%]',
            title='Wpływ R² na pokrycie kalendarza')
for ax in axes: ax.grid(alpha=.25)
fig.tight_layout()
fig.savefig(OUT/'11_03_R2_okna_i_coverage.png',dpi=200)
plt.show()

fig, ax = plt.subplots(figsize=(8,6))
sc = ax.scatter(sensitivity.recall, sensitivity.precision,
                c=sensitivity.coverage_pct, cmap='viridis', s=50, alpha=.85)
ax.scatter([base_row.recall], [base_row.precision], c='red', s=125,
           edgecolors='black', marker='*', label='Filtr bazowy')
ax.set(xlabel='Recall dzienny (UNION)', ylabel='Precision dzienne (UNION)',
       title='Filtry LPPLS: kompromis precision–recall (retrospektywny)')
ax.legend(); ax.grid(alpha=.25)
fig.colorbar(sc,ax=ax,label='Dni z sygnałem [%]')
fig.tight_layout()
fig.savefig(OUT/'11_04_tradeoff_filtracji.png',dpi=200)
plt.show()

## 6. Dzienny Confidence Indicator i historia epizodów

In [ ]:
base_ci = np.divide(base_counts,n_all,out=np.zeros(len(dates),dtype=float),where=n_all>0)
daily = pd.DataFrame(dict(date=dates, n_all=n_all, n_accepted=base_counts,
                          confidence=base_ci, alarm=base_days.astype(int),
                          literature_bubble=reference))
daily.to_csv(OUT/'11_05_daily_CI_baseline.csv',index=False)

fig, ax = plt.subplots(figsize=(15,4.5))
ax.plot(dates, base_ci, lw=.8, label='LPPLS Confidence Indicator')
for i, ep in enumerate(episodes.itertuples(index=False)):
    ax.axvspan(ep.start, ep.end, color='salmon', alpha=.18,
               label='Epizod UNION' if i == 0 else None)
ax.set(xlabel='Data',ylabel='CI',title='Historyczne epizody UNION i CI (filtr bazowy)')
ax.grid(alpha=.2);ax.legend(loc='upper left')
fig.tight_layout()
fig.savefig(OUT/'11_06_CI_historia.png',dpi=200)
plt.show()

print('Dni z alarmem, baza:', int(daily.alarm.sum()))
print('Dni bez okien LPPLS (brak możliwości wygenerowania CI):', int((daily.n_all==0).sum()))

## 7. Wyniki roczne i kontrast filtrów: luźny vs bazowy vs ścisły

In [ ]:
representatives = {
    'luźny': 'r2_0.80_w_6_13_tc_40_m_0_1',
    'bazowy': base_id,
    'ścisły': 'r2_0.95_w_6_13_tc_40_m_0_1',
}
compare = sensitivity.set_index('config_id').loc[list(representatives.values())].copy()
compare.insert(0,'wariant',list(representatives.keys()))
cols=['wariant','r2_min','accepted_windows','days_with_signal',
      'precision','recall','f1','bubble_episode_hits','average_precision']
display(compare[cols].round(3))
compare[cols].to_csv(OUT/'11_07_three_filters_comparison.csv',index=False)

annual_compare = annual[annual.config_id.isin(representatives.values())].copy()
annual_compare['wariant'] = annual_compare.config_id.map({v:k for k,v in representatives.items()})
display(annual_compare.loc[:,['year','wariant','alarm_days','FP','TP','precision','recall']].round(3).head(20))

fig, ax = plt.subplots(figsize=(11,4.5))
for label, cid in representatives.items():
    part=annual_compare.loc[annual_compare.config_id.eq(cid)].sort_values('year')
    ax.plot(part.year,part.alarm_days,marker='o',label=label)
ax.set(xlabel='Rok', ylabel='Liczba dni z alarmem',
       title='Dzienna częstość alarmowania w kolejnych latach')
ax.legend();ax.grid(alpha=.2)
fig.tight_layout()
fig.savefig(OUT/'11_08_alarm_days_per_year.png',dpi=200)
plt.show()

## 8. Opcjonalny test: stabilność losowych inicjalizacji

W pliku CSV jest **tylko jedno dopasowanie dla każdego okna**. Zatem rozrzut między różnymi oknami nie jest miarą stabilności względem inicjalizacji. Gdy ustawisz `RUN_REFIT_STABILITY = True`, poniższy kod wyznaczy kilka nowych dopasowań **tych samych** okien przy różnych seedach.

Do uruchomienia wymagane: `btc_usd_2015_2024.csv` (`Date`, `price`) i pakiet `lppls` (`pip install lppls`). Liczba okien i ziaren jest celowo niewielka, bo estymacja jest kosztowna. Wyniki można rozszerzyć po pilotażu.

In [ ]:
if not RUN_REFIT_STABILITY:
    print('Powtórne dopasowania wyłączone — brak danych o stabilności inicjalizacji.')
else:
    import random
    from lppls import lppls

    if not PRICE_FILE.exists():
        raise FileNotFoundError('Włączono refit, ale brak btc_usd_2015_2024.csv')
    prices = pd.read_csv(PRICE_FILE)
    if 'Date' not in prices.columns or 'price' not in prices.columns:
        raise ValueError('Plik cen musi zawierać Date i price')
    prices['Date'] = pd.to_datetime(prices['Date']).dt.normalize()
    prices = prices.sort_values('Date').drop_duplicates('Date').set_index('Date')
    prices['price'] = pd.to_numeric(prices.price,errors='raise')
    if prices.price.isna().any() or (prices.price <= 0).any():
        raise ValueError('Nieprawidłowe ceny BTC')

    rng = np.random.default_rng(2026)
    original = fits.reset_index(drop=True).copy()
    original['base_accepted'] = base_mask
    selected = []
    for outcome in [True,False]:
        pool = original.index[original.base_accepted.eq(outcome)].to_numpy()
        amount = min(REFIT_WINDOWS_PER_CLASS, len(pool))
        selected.extend(rng.choice(pool, size=amount, replace=False).tolist())

    rows=[]
    for index in selected:
        source_row=original.iloc[index]
        ds=pd.date_range(source_row.win_start,source_row.win_end,freq='D')
        if len(ds) != int(source_row.n) or not prices.index.isin(ds).any():
            raise ValueError('Niespójna długość wybranego okna')
        block = prices.reindex(ds)
        if block.price.isna().any():
            raise ValueError(f'Brakuje cen dla okna {source_row.win_start}-{source_row.win_end}')
        t=np.array([x.toordinal() for x in ds],dtype=float)
        y=np.log(block.price.to_numpy(dtype=float))
        rows.append(dict(window_id=index, seed='original',
                         win_start=source_row.win_start,win_end=source_row.win_end,
                         ok=bool(source_row.ok), t_gap=float(source_row.t_gap),
                         r2=float(source_row.r2), m=float(source_row.m),
                         w=float(source_row.w), B=float(source_row.B),
                         accepted=bool(source_row.base_accepted),error=''))
        for seed in REPEAT_SEEDS:
            random.seed(seed)
            np.random.seed(seed)
            try:
                model=lppls.LPPLS(observations=np.vstack([t,y]))
                tc,m,w,A,B,C,c1,c2,O,D=model.fit(REFIT_MAX_SEARCHES)
                curve=np.asarray(model.lppls(t,tc,m,w,A,B,c1,c2),dtype=float)
                sst=np.sum((y-y.mean())**2)
                r2=1-np.sum((y-curve)**2)/sst if sst>0 else np.nan
                gap=float(tc-t[-1])
                accepted=(0<m<1 and 6<=w<=13 and B<0 and 0<gap<=40 and r2>0.90)
                result=dict(window_id=index,seed=seed,
                            win_start=source_row.win_start,win_end=source_row.win_end,
                            ok=True,t_gap=gap,r2=r2,m=m,w=w,B=B,
                            accepted=accepted,error='')
            except Exception as e:
                result=dict(window_id=index,seed=seed,
                            win_start=source_row.win_start,win_end=source_row.win_end,
                            ok=False,t_gap=np.nan,r2=np.nan,m=np.nan,
                            w=np.nan,B=np.nan,accepted=False,error=str(e)[:200])
            rows.append(result)

    repeated=pd.DataFrame(rows)
    repeated.to_csv(OUT/'11_09_refit_by_seed.csv',index=False)
    refits=repeated.loc[repeated.seed.ne('original')].copy()
    stability=refits.groupby('window_id').agg(
        n_runs=('seed','size'), successful=('ok','sum'),
        acceptance_rate=('accepted','mean'),
        t_gap_std=('t_gap','std'), t_gap_median=('t_gap','median'),
        omega_std=('w','std'), m_std=('m','std'),
        r2_median=('r2','median')
    ).reset_index()
    originals=repeated.loc[repeated.seed.eq('original'),['window_id','accepted']].rename(
        columns={'accepted':'original_accepted'})
    stability=stability.merge(originals,on='window_id')
    stability.to_csv(OUT/'11_10_stability_by_window.csv',index=False)
    print('Powtórnie ocenione okna:',len(stability))
    display(stability.sort_values('t_gap_std',ascending=False).head(10).round(3))

## 9. Podsumowanie i wskazówki do publikacji

**Wnioski z tej analizy mogą być następujące:**

1. Jak restrykcyjne filtry wpływają na odsetek zaakceptowanych dopasowań oraz pokrycie kalendarzowe.
2. Jaki jest kompromis między liczbą alarmów a zgodnością z historycznymi epizodami. Pamiętaj: `UNION` to **retrospektywne etykiety** i nie są niezależne od metod typu LPPLS.
3. Czy zaobserwowane sygnały są stabilne między latami.
4. Po **włączeniu ponownych dopasowań**: jak duża jest zmienność oszacowanego `t_gap`, `m`, `ω` i decyzji akceptacyjnych przy różnych inicjalizacjach.

Nie wybieraj najlepszego filtra po wynikach na całym 2015–2024, a następnie nie prezentuj go jako niezależnej jakości OOS. Do odpowiedzi na pytanie, czy zmiana filtrów **poprawia prognozowanie** spadków, trzeba ponownie wyznaczyć cechy dzienne i powtórzyć trenowanie modeli z zamrożonymi granicami walk-forward.

In [ ]:
print('=== ZESTAWIENIE ===')
print('Akceptacja bazowa:', int(base_mask.sum()), '/', len(fits))
print('Pokrycie dni:', int(base_days.sum()), '/', len(dates))
print('Konfiguracje filtrów:',len(sensitivity))
print('Zapisane pliki:')
for p in sorted(OUT.glob('11_*')):
    print(' -',p.name)